# GÜN 154: Protocol Buffers (Protobuf) Zihinsel Modeli ve Tel Formatı (Wire Format)

**Müfredat:** 365 Günlük ASELSAN / HAVELSAN Savunma Sistemleri C++20 & Linux Sistem Mühendisliği  
**Faz:** F4 — TCP/UDP / Ağ Protokolleri / Serileştirme & Taktik Dağıtık Sistemler  
**Konu:** Google Protocol Buffers (Protobuf) Mimarisi, Şema Tabanlı Kod Üretimi (`.proto`), Tel Formatı (Wire Format), Alan Numaralandırma (Field Tagging), Varint (Variable-length Quantity) Kodlaması, ZigZag Kodlama, Tel Tipleri (Wire Types 0, 1, 2, 5), Şema Evrimi (Schema Evolution) ve C++20 Taktik Dağıtık Sistem Entegrasyonu  
**Ön Koşul:** Gün 153 (Serialization: JSON vs binary)

---

### Telif ve Yasal Uyarı
```text
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)

Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

## 1. Zihinsel Model & Savunma Sanayii Konsepti

Hava savunma füzeleri, radar ağları, insansız araç filoları ve C4ISR komuta kontrol merkezleri onlarca farklı programlama dili (C++, Python, Rust, Java, C#) ve farklı işletim sistemleri (Linux, VxWorks, QNX, Windows) üzerinde çalışır:
- C++ ile yazılmış bir radar izleme motoru,
- Python ile geliştirilmiş bir yapay zeka hedef sınıflandırma modülü,
- Web ve Java tabanlı bir harekat merkezi harita sunucusu
aynı taktik veriyi paylaşmak zorundadır.

Eğer C-struct (`memcpy`) kullanırsanız; diller arası veri tipi farkları, derleyicilerin yapı dolguları (struct padding) ve bellek hizalamaları (alignment) sistemi kilitler.  
Eğer JSON kullanırsanız; bant genişliği %500 şişer ve mikrosaniyeler mertebesinde string ayrıştırma gecikmesi yaşanır.

Google tarafından geliştirilen **Protocol Buffers (Protobuf)**, savunma sanayiinde bu iki dünyanın mükemmel kesişim noktasını oluşturur:
1. **Şema Sözleşmesi (Schema-First Contract):** Veri `.proto` uzantılı bağımsız bir dosyada tanımlanır.
2. **Otomatik Kod Üretimi:** `protoc` derleyicisi bu dosyadan her hedef dil için yüksek performanslı, tip-güvenli sınıflar üretir.
3. **Kompakt Tel Formatı (Wire Format):** Telde alan adları (`"target_azimuth"`) yerine küçük tamsayı etiketleri (Field Tags) ve değişken uzunluklu tamsayılar (**Varint**) taşınır.

```
   +-------------------------------------------------------------------------+
   |                PROTOBUF ZİHİNSEL MODELİ VE ÇALIŞMA PRENSİBİ             |
   +-------------------------------------------------------------------------+

      [ target.proto ]  (Taktik Şema Dosyası)
            |
      +-----+-----+
      |  protoc   |  (Protocol Compiler)
      +-----+-----+
            |-----------------------------+-----------------------------+
            v                             v                             v
      [ target.pb.h / .cc ]        [ target_pb2.py ]            [ Target.java ]
     (C++20 Radar Motoru)        (Python AI Sınıflandırıcı)    (C2 Karargah Haritası)
            |                             |                             |
            +-----------------------------+-----------------------------+
                                          |
                        [ Teldeki İkili Akış (Wire Stream) ]
                        Tag(1) + Varint + Tag(2) + Fixed32 ...
                        (Platform Bağımsız, Minimum Boyut, %100 Uyum)
```

## 2. Protobuf Tel Formatı (Wire Format Anatomisi)

Protobuf mesajları tel üzerinde ardışık anahtar-değer (Key-Value) çiftleri olarak akar. Her alan bir **Key (Tag)** ile başlar:

$$\text{Key (Tag)} = (\text{Field Number} \ll 3) \mid \text{Wire Type}$$

En alttaki 3 bit **Wire Type**'ı belirtir; geri kalan üst bitler ise `.proto` dosyasındaki **Field Number**'dır:

| Wire Type | İsim | Temsil Edilen Tipler | Teldeki Format |
| :--- | :--- | :--- | :--- |
| **0** | **Varint** | `int32`, `int64`, `uint32`, `uint64`, `sint32`, `sint64`, `bool`, `enum` | Değişken uzunluklu (1-10 bayt) |
| **1** | **64-bit** | `fixed64`, `sfixed64`, `double` | Sabit 8 bayt (Little-Endian) |
| **2** | **Length-delimited**| `string`, `bytes`, embedded messages, packed repeated | Varint Length + Raw Bytes |
| **5** | **32-bit** | `fixed32`, `sfixed32`, `float` | Sabit 4 bayt (Little-Endian) |

### Tag Hesaplama Örneği:
- Alan Tanımı: `uint32 target_id = 1;`
  - Field Number = 1, Wire Type = 0 (Varint).
  - $\text{Key} = (1 \ll 3) \mid 0 = 00001000_2 = \mathbf{0x08}$.
- Alan Tanımı: `float azimuth = 2;`
  - Field Number = 2, Wire Type = 5 (32-bit fixed).
  - $\text{Key} = (2 \ll 3) \mid 5 = 00010101_2 = \mathbf{0x15}$.

## 3. Varint ve ZigZag Kodlama Matematiği

### 1. Varint (Variable-length Quantity - Değişken Uzunluklu Tamsayı)
Varint, bir tamsayıyı değerinin büyüklüğüne göre 1 ila 10 bayt arasında saklar:
- Her baytın **7 biti** gerçek veriyi taşır.
- En yüksek anlamlı biti (MSB - Most Significant Bit) **Devam Bayrağı (Continuation Bit)** olarak kullanılır:
  - $\text{MSB} = 1$: Sırada başka baytlar var demektir.
  - $\text{MSB} = 0$: Varint dizisinin son baytıdır.

#### Örnek: `42` Sayısının Kodlanması:
- $42 = 00101010_2$. 7 bite sığar. MSB = 0 $\to \mathbf{0x2A}$ (Yalnızca 1 bayt!).
- Normal 32-bit integer bellekte 4 bayt iken, Protobuf telinde **1 bayta** iner (%75 tasarruf).

### 2. ZigZag Kodlaması (`sint32` / `sint64`)
Normal `int32` tipinde $-1$ sayısı ikiye tümleyen (two's complement) gösteriminde $0xFFFFFFFF$ olduğundan, Varint kodlamasında tam **10 bayt** yer kaplar! Bu durum negatif koordinat ve hızlarda felakettir.  
Protobuf bu sorunu **ZigZag Kodlaması** ile çözer:
- Negatif ve pozitif sayılar pozitif tamsayılara çapraz eşlenir:
  $$0 \to 0,\quad -1 \to 1,\quad 1 \to 2,\quad -2 \to 3,\quad 2 \to 4,\quad -3 \to 5$$
- Matematiksel Formül:
  $$\text{ZigZag32}(n) = (n \ll 1) \oplus (n \gg 31)$$
Böylece $-1$ sayısı telde $1$ olarak kodlanır ve 10 bayt yerine **1 bayta** düşer!

## 4. Şema Evrimi ve İleri/Geriye Dönük Uyumluluk Kuralları

Savunma sistemlerinde Protobuf'ın sağladığı geriye ve ileriye dönük uyumluluk kuralları:

### 1. Bilinmeyen Alanların Korunması (Unknown Fields Preservation):
Eğer eski bir C2 yazılımı (v1), modern bir radarın (v2) gönderdiği yeni alanı (`ai_threat_score = 4`) tanımıyorsa:
- Wire Type'a bakar (örn: Type 0 ise Varint'i sonuna kadar okur, Type 2 ise Length kadar atlar).
- Bilinmeyen alanı çökmeksizin güvenle atlar (**Postel Kanunu**).
- Gerekirse bu bilinmeyen baytları bellekte saklar ve mesajı başka bir yere iletirken kaybetmeden geri yazar!

### 2. Altın Kurallar (Schema Evolution Golden Rules):
1. **Alan Numarasını (Field Number) Asla Değiştirme:** İsimler değişebilir (`target_id` yerine `radar_track_id`), ancak numara (`= 1`) teldeki tek kimliktir!
2. **Kaldırılan Alan Numaralarını Rezerve Et:**
   ```protobuf
   message RadarTrack {
       reserved 3, 7, 10 to 15; // Gelecekte yanlışlıkla tekrar kullanılamaz!
       reserved "deprecated_elevation";
   }
   ```
3. **Proto3 Varsayılan Değerleri:** Mesajda eksik olan alanlar için alıcı tarafında otomatik olarak varsayılan değer atanır (`numeric = 0`, `string = ""`, `bool = false`). Telde sıfır olan alanlar gönderilmez (Sıfır tasarrufu).

## 5. C++20 ile Protobuf Entegrasyonu ve Performans İlkeleri

Modern C++20 projelerinde Protobuf kullanılırken dikkat edilmesi gereken mimari noktalar:

### 1. SerializeToArray vs SerializeToString:
```cpp
// YANLIŞ: std::string dinamik heap tahsisi (malloc) yapar, determinizmi bozar
std::string payload;
track.SerializeToString(&payload);

// DOĞRU (Savunma Standardı): Önceden ayrılmış stack/static tampona doğrudan serileştirme
std::array<uint8_t, 256> buffer;
if (track.SerializeToArray(buffer.data(), buffer.size())) {
    send_to_radio(std::span{buffer.data(), (size_t)track.ByteSizeLong()});
}
```

### 2. Arena Allocation (Google Protobuf Arena):
Yüksek frekanslı (1000 Hz) mesaj döngülerinde `malloc`/`free` gecikmesini önlemek için Protobuf'ın **Arena Bellek Tahsisçisi** kullanılmalıdır:
```cpp
google::protobuf::Arena arena;
RadarTrack* track = google::protobuf::Arena::CreateMessage<RadarTrack>(&arena);
// Tüm iç nesneler tek bir bellek bloğunda açılır, arena resetlendiğinde anında serbest kalır!
```

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 1 — Protobuf Tel Formatı (Wire Engine): Varint, ZigZag ve Tag Motoru
"""

import struct
from typing import Tuple, List, Dict, Any

class ProtobufWire:
    """
    Google Protocol Buffers tel formatı (Wire Format) kodlama ve çözme motoru.
    """
    WIRE_VARINT = 0
    WIRE_FIXED64 = 1
    WIRE_LENGTH_DELIMITED = 2
    WIRE_FIXED32 = 5

    # 1. Varint Kodlama ve Çözme
    @staticmethod
    def encode_varint(value: int) -> bytes:
        if value < 0:
            # 64-bit two's complement
            value = (1 << 64) + value
        out = bytearray()
        while value > 0x7F:
            out.append((value & 0x7F) | 0x80)
            value >>= 7
        out.append(value & 0x7F)
        return bytes(out)

    @staticmethod
    def decode_varint(data: bytes, offset: int = 0) -> Tuple[int, int]:
        res = 0
        shift = 0
        while offset < len(data):
            b = data[offset]
            offset += 1
            res |= (b & 0x7F) << shift
            if not (b & 0x80):
                break
            shift += 7
        return res, offset

    # 2. ZigZag Kodlama (sint32 için)
    @staticmethod
    def encode_zigzag32(n: int) -> int:
        return (n << 1) ^ (n >> 31)

    @staticmethod
    def decode_zigzag32(n: int) -> int:
        return (n >> 1) ^ (-(n & 1))

    # 3. Tag (Key) Oluşturma ve Çözme
    @classmethod
    def make_tag(cls, field_number: int, wire_type: int) -> bytes:
        key = (field_number << 3) | (wire_type & 0x07)
        return cls.encode_varint(key)

    @classmethod
    def parse_tag(cls, data: bytes, offset: int = 0) -> Tuple[int, int, int]:
        key, new_offset = cls.decode_varint(data, offset)
        field_number = key >> 3
        wire_type = key & 0x07
        return field_number, wire_type, new_offset

print("[HÜCRE 7] Protobuf Tel Formatı Motoru Başarıyla Yüklendi.")

# Temel Varint ve ZigZag Doğrulamaları
v_test = 300
v_enc = ProtobufWire.encode_varint(v_test)
v_dec, _ = ProtobufWire.decode_varint(v_enc)

z_test = -42
z_enc = ProtobufWire.encode_zigzag32(z_test)
z_dec = ProtobufWire.decode_zigzag32(z_enc)

print(f"  [+] Sayı 300 Varint Baytları : {[hex(b) for b in v_enc]} ({len(v_enc)} Bayt)")
print(f"  [+] Varint Geri Çözme        : {v_dec} (Doğrulandı: {v_dec == v_test})")
print(f"  [+] Sayı -42 ZigZag Değeri   : {z_enc} (Pozitife eşlendi)")
print(f"  [+] ZigZag Geri Çözme        : {z_dec} (Doğrulandı: {z_dec == z_test})")

tag_id = ProtobufWire.make_tag(field_number=1, wire_type=ProtobufWire.WIRE_VARINT)
tag_az = ProtobufWire.make_tag(field_number=2, wire_type=ProtobufWire.WIRE_FIXED32)
print(f"  [+] Field 1 Varint Tag Hex  : 0x{tag_id.hex().upper()} (Beklenen: 0x08)")
print(f"  [+] Field 2 Fixed32 Tag Hex : 0x{tag_az.hex().upper()} (Beklenen: 0x15)")

## 6. Taktik Radar İzi (RadarTrack.proto) Mesaj Modeli

Bu senaryoda bir taktik hava savunma radarı için `.proto` şeması modellenmiştir:

```protobuf
syntax = "proto3";
package tactical;

message RadarTrack {
    uint32 track_id    = 1; // Wire Type 0 (Varint)
    float  azimuth     = 2; // Wire Type 5 (Fixed32)
    float  elevation   = 3; // Wire Type 5 (Fixed32)
    float  range_m     = 4; // Wire Type 5 (Fixed32)
    sint32 radial_vel  = 5; // Wire Type 0 (Varint / ZigZag)
    string classification = 6; // Wire Type 2 (Length-delimited)
}
```

Aşağıdaki kod hücresinde, bu şemanın Protobuf ikili tel formatına nasıl dönüştürüldüğünü ve teldeki her baytın anlamını adım adım ayrıştıracağız.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 2 — Taktik Radar İzi Protobuf Mesaj Serileştirici ve Ayrıştırıcı
"""

class RadarTrackMessage:
    def __init__(self, track_id: int, azimuth: float, elevation: float, range_m: float, radial_vel: int, classification: str):
        self.track_id = track_id
        self.azimuth = azimuth
        self.elevation = elevation
        self.range_m = range_m
        self.radial_vel = radial_vel # Yaklaşma/Uzaklaşma hızı (m/s, negatif olabilir)
        self.classification = classification

    def serialize_to_wire(self) -> bytes:
        out = bytearray()
        
        # Field 1: track_id (uint32 -> Varint)
        if self.track_id != 0:
            out.extend(ProtobufWire.make_tag(1, ProtobufWire.WIRE_VARINT))
            out.extend(ProtobufWire.encode_varint(self.track_id))

        # Field 2: azimuth (float -> Fixed32)
        if self.azimuth != 0.0:
            out.extend(ProtobufWire.make_tag(2, ProtobufWire.WIRE_FIXED32))
            out.extend(struct.pack("<f", self.azimuth)) # Protobuf fixed types are Little-Endian

        # Field 3: elevation (float -> Fixed32)
        if self.elevation != 0.0:
            out.extend(ProtobufWire.make_tag(3, ProtobufWire.WIRE_FIXED32))
            out.extend(struct.pack("<f", self.elevation))

        # Field 4: range_m (float -> Fixed32)
        if self.range_m != 0.0:
            out.extend(ProtobufWire.make_tag(4, ProtobufWire.WIRE_FIXED32))
            out.extend(struct.pack("<f", self.range_m))

        # Field 5: radial_vel (sint32 -> ZigZag Varint)
        if self.radial_vel != 0:
            out.extend(ProtobufWire.make_tag(5, ProtobufWire.WIRE_VARINT))
            zz = ProtobufWire.encode_zigzag32(self.radial_vel)
            out.extend(ProtobufWire.encode_varint(zz))

        # Field 6: classification (string -> Length-delimited)
        if self.classification:
            str_bytes = self.classification.encode('utf-8')
            out.extend(ProtobufWire.make_tag(6, ProtobufWire.WIRE_LENGTH_DELIMITED))
            out.extend(ProtobufWire.encode_varint(len(str_bytes)))
            out.extend(str_bytes)

        return bytes(out)

    @classmethod
    def parse_from_wire(cls, wire_bytes: bytes) -> Tuple['RadarTrackMessage', Dict[int, Any]]:
        offset = 0
        track_id = 0
        azimuth = 0.0
        elevation = 0.0
        range_m = 0.0
        radial_vel = 0
        classification = ""
        unknown_fields = {}

        while offset < len(wire_bytes):
            field_num, wire_type, offset = ProtobufWire.parse_tag(wire_bytes, offset)
            
            if field_num == 1 and wire_type == ProtobufWire.WIRE_VARINT:
                track_id, offset = ProtobufWire.decode_varint(wire_bytes, offset)
            elif field_num == 2 and wire_type == ProtobufWire.WIRE_FIXED32:
                azimuth = struct.unpack("<f", wire_bytes[offset:offset+4])[0]
                offset += 4
            elif field_num == 3 and wire_type == ProtobufWire.WIRE_FIXED32:
                elevation = struct.unpack("<f", wire_bytes[offset:offset+4])[0]
                offset += 4
            elif field_num == 4 and wire_type == ProtobufWire.WIRE_FIXED32:
                range_m = struct.unpack("<f", wire_bytes[offset:offset+4])[0]
                offset += 4
            elif field_num == 5 and wire_type == ProtobufWire.WIRE_VARINT:
                zz, offset = ProtobufWire.decode_varint(wire_bytes, offset)
                radial_vel = ProtobufWire.decode_zigzag32(zz)
            elif field_num == 6 and wire_type == ProtobufWire.WIRE_LENGTH_DELIMITED:
                length, offset = ProtobufWire.decode_varint(wire_bytes, offset)
                classification = wire_bytes[offset:offset+length].decode('utf-8')
                offset += length
            else:
                # BİLİNMEYEN ALAN (FORWARD COMPATIBILITY / POSTEL ATLATMA)
                if wire_type == ProtobufWire.WIRE_VARINT:
                    val, offset = ProtobufWire.decode_varint(wire_bytes, offset)
                    unknown_fields[field_num] = val
                elif wire_type == ProtobufWire.WIRE_FIXED32:
                    unknown_fields[field_num] = wire_bytes[offset:offset+4]
                    offset += 4
                elif wire_type == ProtobufWire.WIRE_FIXED64:
                    unknown_fields[field_num] = wire_bytes[offset:offset+8]
                    offset += 8
                elif wire_type == ProtobufWire.WIRE_LENGTH_DELIMITED:
                    length, offset = ProtobufWire.decode_varint(wire_bytes, offset)
                    unknown_fields[field_num] = wire_bytes[offset:offset+length]
                    offset += length

        msg = cls(track_id, azimuth, elevation, range_m, radial_vel, classification)
        return msg, unknown_fields

# Test Radar İzi Oluştur
track_obj = RadarTrackMessage(
    track_id=1042,
    azimuth=88.5,
    elevation=12.3,
    range_m=45200.0,
    radial_vel=-350, # Bize doğru 350 m/s hızla yaklaşan düşman hedefi
    classification="SU-57"
)

wire_data = track_obj.serialize_to_wire()
parsed_track, unknowns = RadarTrackMessage.parse_from_wire(wire_data)

print()
print("[*] Protobuf RadarTrack Serileştirme Raporu:")
print(f"  [+] Teldeki Toplam Boyut  : {len(wire_data)} Bayt")
print(f"  [+] Teldeki Hex Akışı     : {wire_data.hex().upper()}")
print(f"  [+] Çözülen Hedef ID      : {parsed_track.track_id}")
print(f"  [+] Çözülen Azimut Açısı  : {parsed_track.azimuth:.1f}°")
print(f"  [+] Çözülen Radyal Hız    : {parsed_track.radial_vel} m/s (Negatif değer korundu)")
print(f"  [+] Çözülen Sınıflandırma : {parsed_track.classification}")

assert parsed_track.track_id == 1042
assert parsed_track.radial_vel == -350
assert parsed_track.classification == "SU-57"
assert len(unknowns) == 0
print("[HÜCRE 9] RadarTrack Protobuf tel formatı testi başarıyla tamamlandı.")

## 7. Bilerek Bozma (Sabotaj ve Arıza Enjeksiyonu)

Protobuf mimarisinde en sık yapılan iki kritik mühendislik hatasını test edeceğiz:

### Sabotaj 1: Negatif Sayılarda Düz `int32` vs `sint32` (Tel Şişme Tuzağı)
Geliştirici radar yaklaşma hızını (`radial_vel = -350`) tanımlarken `sint32` yerine standart `int32` kullanmıştır.
- İkiye tümleyen negatif sayılar Varint'e sokulduğunda 64-bite işaret genişletmesi (sign extension) yapılır ve tam **10 bayt** tel kaplar!
- `sint32` ZigZag kullanıldığında ise $-350$ sayısı pozitif $699$ sayısına eşlenir ve yalnızca **2 bayt** tutar!
- Bu hata taktik telsiz kanalında gereksiz yere %500 ek bayt taşınmasına sebep olur.

### Sabotaj 2: Bilinmeyen Alan (Unknown Field Tag 9) ve İleriye Dönük Uyumluluk
Gelecekte üretilen bir AESA radar yazılımı, pakete `Tag 9` ile 8 baytlık `Fixed64` formatında yeni bir elektronik harp (EW) parametresi ekler.
- Eski sürüm alıcısının bu paketi çökmeksizin ayrıştırması ve bilinmeyen Tag 9'u güvenle `unknown_fields` sözlüğünde saklaması doğrulanacaktır.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 3 — Sabotaj Deneyleri: Negatif int32 Şişmesi ve Bilinmeyen Alan
"""

print()
print("[*] Protobuf Sabotaj ve İleriye Uyumluluk Testleri Başlatılıyor...")

# -----------------------------------------------------------------------------
# SABOTAJ 1: int32 (-350) vs sint32 (-350) Tel Boyutu Karşılaştırması
# -----------------------------------------------------------------------------
neg_val = -350

# A) Hatalı Yaklaşım: Standart int32 Two's Complement Varint
raw_int32_varint = ProtobufWire.encode_varint(neg_val)

# B) Doğru Yaklaşım: sint32 ZigZag Varint
zz_val = ProtobufWire.encode_zigzag32(neg_val)
zigzag_varint = ProtobufWire.encode_varint(zz_val)

print(f"  [+] Sabotaj 1 Sonuçları (Negatif Sayı Tel Şişmesi):")
print(f"      - int32 (-350) Varint Boyutu : {len(raw_int32_varint)} Bayt (ŞİŞME! Tam 10 Bayt!)")
print(f"      - sint32 (-350) ZigZag Boyut : {len(zigzag_varint)} Bayt (MÜKEMMEL! 2 Bayt!)")
print(f"      - İsraf Oranı               : {len(raw_int32_varint) / len(zigzag_varint):.1f}x Daha Fazla Bayt!")

assert len(raw_int32_varint) == 10, "int32 negatif sayı 10 bayta genişlemelidir!"
assert len(zigzag_varint) == 2, "sint32 negatif sayı 2 bayt tutmalıdır!"

# -----------------------------------------------------------------------------
# SABOTAJ 2: Bilinmeyen Alan (Tag 9 / Fixed64) İle İleriye Uyumluluk Testi
# -----------------------------------------------------------------------------
# Yeni Nesil Radar, eski RadarTrack mesajının sonuna Tag 9 (Fixed64) Elektronik Harp verisi ekledi:
future_wire = bytearray(wire_data)
tag_future = ProtobufWire.make_tag(field_number=9, wire_type=ProtobufWire.WIRE_FIXED64)
future_data = struct.pack("<Q", 0xDEADBEEFCAFEBABE)

future_wire.extend(tag_future)
future_wire.extend(future_data)

# Eski alıcı bu geleceğin paketini çözebilecek mi?
parsed_legacy, legacy_unknowns = RadarTrackMessage.parse_from_wire(bytes(future_wire))

print()
print(f"  [+] Sabotaj 2 Sonuçları (Bilinmeyen Alan İle Karşılaşma):")
print(f"      - Eski Alıcı Çöktü mü?        : HAYIR (Güvenle Çalıştı)")
print(f"      - Bilinen Alanlar Okundu mu?  : EVET (Hedef: {parsed_legacy.classification})")
print(f"      - Bilinmeyen Tag 9 Yakalandı mı: {9 in legacy_unknowns} (Boyut: {len(legacy_unknowns[9])} Bayt)")

assert 9 in legacy_unknowns
assert legacy_unknowns[9] == future_data
print("[HÜCRE 11] Tüm Protobuf sabotaj deneyleri başarıyla tamamlandı.")

## 8. Performans ve Boyut Benchmarkı (JSON vs Protobuf vs C-Struct)

Bir taktik aviyonik sisteminde $40,000$ adet radar hedef mesajı üzerinde 3 format yarışacaktır:
1. **JSON:** Metin tabanlı (`{"track_id":1042,...}`).
2. **Protocol Buffers (Protobuf):** Varint + Tag ikili akışı.
3. **Raw C-Struct (`memcpy`):** Sabit boyutlu doğal bellek kopyası.

### Karşılaştırma Kriterleri:
- Teldeki Paket Boyutu (Bayt)
- Serileştirme ve Ayrıştırma İşlem Süresi (Nanosaniye / Paket)
- Saniyedeki Paket İşleme Kapasitesi (Throughput)

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 4 — Benchmark: JSON vs Protobuf vs Raw Struct Hız ve Boyut Ölçümü
"""

import time
import json

def run_protobuf_benchmark(iterations: int = 25_000) -> Dict[str, Any]:
    track = RadarTrackMessage(1042, 88.5, 12.3, 45200.0, -350, "SU-57")
    
    # Boyutlar
    proto_bytes = track.serialize_to_wire()
    json_bytes = json.dumps({
        "track_id": track.track_id, "azimuth": track.azimuth,
        "elevation": track.elevation, "range_m": track.range_m,
        "radial_vel": track.radial_vel, "classification": track.classification
    }).encode('utf-8')
    raw_struct_bytes = struct.pack(">I f f f i 8s", track.track_id, track.azimuth, track.elevation, track.range_m, track.radial_vel, track.classification.encode('utf-8'))

    # 1. JSON Süresi
    t0 = time.perf_counter()
    for _ in range(iterations):
        jb = json.dumps({
            "track_id": track.track_id, "azimuth": track.azimuth,
            "elevation": track.elevation, "range_m": track.range_m,
            "radial_vel": track.radial_vel, "classification": track.classification
        }).encode('utf-8')
        _ = json.loads(jb.decode('utf-8'))
    t_json = time.perf_counter() - t0

    # 2. Protobuf Süresi
    t0 = time.perf_counter()
    for _ in range(iterations):
        pb = track.serialize_to_wire()
        _, _ = RadarTrackMessage.parse_from_wire(pb)
    t_proto = time.perf_counter() - t0

    # 3. Raw Struct Süresi
    t0 = time.perf_counter()
    for _ in range(iterations):
        sb = struct.pack(">I f f f i 8s", track.track_id, track.azimuth, track.elevation, track.range_m, track.radial_vel, b"SU-57\x00\x00\x00")
        _ = struct.unpack(">I f f f i 8s", sb)
    t_struct = time.perf_counter() - t0

    return {
        "iterations": iterations,
        "size_json": len(json_bytes),
        "size_proto": len(proto_bytes),
        "size_struct": len(raw_struct_bytes),
        "t_json_ms": t_json * 1000.0,
        "t_proto_ms": t_proto * 1000.0,
        "t_struct_ms": t_struct * 1000.0,
        "json_ns": (t_json / iterations) * 1e9,
        "proto_ns": (t_proto / iterations) * 1e9,
        "struct_ns": (t_struct / iterations) * 1e9,
        "proto_vs_json_speedup": t_json / t_proto
    }

bench_res = run_protobuf_benchmark(iterations=25_000)

print()
print(f"[*] Format Kıyaslama Benchmarkı ({bench_res['iterations']:,} İşlem):")
print(f"  [+] Paket Boyutları     : JSON={bench_res['size_json']}B | Struct={bench_res['size_struct']}B | Protobuf={bench_res['size_proto']}B")
print(f"  [+] JSON Süresi         : {bench_res['t_json_ms']:.2f} ms ({bench_res['json_ns']:.1f} ns/op)")
print(f"  [+] Protobuf Süresi     : {bench_res['t_proto_ms']:.2f} ms ({bench_res['proto_ns']:.1f} ns/op)")
print(f"  [+] Raw Struct Süresi   : {bench_res['t_struct_ms']:.2f} ms ({bench_res['struct_ns']:.1f} ns/op)")
print(f"  [+] Protobuf Kazancı    : JSON'dan {bench_res['size_json'] / bench_res['size_proto']:.1f}x Daha Kompakt!")

assert bench_res['size_proto'] < bench_res['size_json']
print("[HÜCRE 13] Format karşılaştırma benchmarkı başarıyla tamamlandı.")

## 9. Gerçek Dünya Taktik Senaryosu: gRPC ve Askeri C2 Dağıtık Servisi

Protobuf yalnızca veri saklamaz; Google'ın **gRPC** protokolü ile birlikte taktik servislerin omurgasını oluşturur:
- **RPC Metot Tanımı:**
  ```protobuf
  service MissileGuidanceService {
      rpc StreamRadarTracks (RadarStreamRequest) returns (stream RadarTrack);
      rpc EngageTarget (FireCommand) returns (FireResponse);
  }
  ```
- **Taktik Avantajı:** HTTP/2 tabanlı ikili akış (Binary Streaming), tek bir TCP soketi üzerinden çoklanmış (multiplexed) çift yönlü haberleşme sunar.
- Aşağıdaki kod hücresinde, taktik bir radar akışını Protobuf tabanlı bir yayıncı-abone (Pub-Sub) dağıtıcısı üzerinde test edeceğiz.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 5 — Taktik Protobuf Dağıtıcı (Tactical Pub-Sub Broker)
"""

class TacticalProtobufBroker:
    """
    Protobuf ile serileştirilmiş radar izlerini abonelere dağıtan servis.
    """
    def __init__(self):
        self.published_messages = 0
        self.delivered_tracks = 0
        self.total_wire_bytes = 0

    def publish_radar_track(self, track_msg: RadarTrackMessage) -> bytes:
        wire_data = track_msg.serialize_to_wire()
        self.published_messages += 1
        self.total_wire_bytes += len(wire_data)
        return wire_data

    def deliver_to_subscriber(self, wire_data: bytes) -> RadarTrackMessage:
        parsed, _ = RadarTrackMessage.parse_from_wire(wire_data)
        self.delivered_tracks += 1
        return parsed

broker = TacticalProtobufBroker()

# 5 Hedeflik Radar Taramasını Yayınla ve Dağıt
fleet_tracks = [
    RadarTrackMessage(101, 45.0, 10.0, 15000.0, -280, "UAV"),
    RadarTrackMessage(102, 90.0, 15.0, 32000.0, -650, "MISSILE"),
    RadarTrackMessage(103, 135.0, 5.0, 48000.0, 120, "AIRLINER")
]

delivered_list = []
for t in fleet_tracks:
    w = broker.publish_radar_track(t)
    rcv = broker.deliver_to_subscriber(w)
    delivered_list.append(rcv)

print()
print("[*] Taktik Protobuf Dağıtıcı (Broker) Raporu:")
print(f"  [+] Yayınlanan Mesaj Sayısı : {broker.published_messages}")
print(f"  [+] Teslim Edilen İzler     : {broker.delivered_tracks}")
print(f"  [+] Toplam Taşınan Bayt     : {broker.total_wire_bytes} Bayt")
print(f"  [+] Tehdit Sınıfı           : {[t.classification for t in delivered_list]}")

assert broker.delivered_tracks == 3
assert delivered_list[1].classification == "MISSILE"
assert delivered_list[1].radial_vel == -650
print("[HÜCRE 15] Taktik Protobuf dağıtıcı testi başarıyla tamamlandı.")

## 10. 4-Panelli 300 DPI Görselleştirme Mimarisi

Aşağıdaki kod hücresi; Protobuf tel formatı anatomisini, Varint/ZigZag bit haritasını, şema evrimi ve bilinmeyen alan atlatma mekanizmasını ve format kıyaslama benchmarkını 4 panelde detaylandıran profesyonel bir karanlık tema (`#0F172A`) görsel panosu üretir:

1. **Panel 1 (Sol Üst): Protobuf Tel Formatı (Wire Format Anatomisi):**
   Tag `(Field Number << 3 | Wire Type)` ve ardından gelen Varint / Fixed32 değerlerinin bayt blok şeması.
2. **Panel 2 (Sağ Üst): Varint vs ZigZag Kodlama Verimliliği:**
   Pozitif sayılarda ve negatif sayılarda (`int32` vs `sint32`) bayt boyutu tasarruf analizi.
3. **Panel 3 (Sol Alt): Şema Evrimi ve Bilinmeyen Alan (Unknown Field 9) Atlatma:**
   Eski alıcının yeni alan karşısında çökmeden `Wire Type` ile güvenle ilerlemesi.
4. **Panel 4 (Sağ Alt): Format Karşılaştırma Benchmarkı (Tel Boyutu ve Gecikme):**
   JSON, Raw Struct ve Protobuf formatlarının bayt ve gecikme metrikleri.

In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 154: Kod Bloğu 6 — 4-Panelli 300 DPI Görselleştirme Panosu ve Katı Doğrulama Testleri
"""

import os
import matplotlib.pyplot as plt
import numpy as np

if 'PNG_FILE' not in globals():
    PNG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "gun_154_protocol_buffers_zihinsel_modeli.png")

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.linewidth'] = 1.2

fig, axs = plt.subplots(2, 2, figsize=(18, 12), dpi=300, facecolor='#0F172A')

# -----------------------------------------------------------------------------
# PANEL 1: Protobuf Tel Formatı (Wire Format Anatomisi)
# -----------------------------------------------------------------------------
ax1 = axs[0, 0]
ax1.set_facecolor('#1E293B')
ax1.axis('off')

wire_boxes = [
    {"name": "Tag 1 (0x08)\nField:1 | Type:0 (Varint)", "x": 0.05, "y": 0.60, "w": 0.22, "c": "#3B82F6"},
    {"name": "track_id (1042)\nVarint: 0x92 0x08", "x": 0.30, "y": 0.60, "w": 0.22, "c": "#10B981"},
    {"name": "Tag 2 (0x15)\nField:2 | Type:5 (Fixed32)", "x": 0.55, "y": 0.60, "w": 0.22, "c": "#3B82F6"},
    {"name": "azimuth (88.5°)\nFixed32: 4 Bayt Little-Endian", "x": 0.80, "y": 0.60, "w": 0.17, "c": "#F59E0B"},
    
    {"name": "Tag 5 (0x28)\nField:5 | Type:0 (Varint)", "x": 0.05, "y": 0.20, "w": 0.22, "c": "#3B82F6"},
    {"name": "radial_vel (-350)\nZigZag: 0xBB 0x05", "x": 0.30, "y": 0.20, "w": 0.22, "c": "#EC4899"},
    {"name": "Tag 6 (0x32)\nField:6 | Type:2 (Delimited)", "x": 0.55, "y": 0.20, "w": 0.22, "c": "#3B82F6"},
    {"name": "classification\nLen: 5 | Value: 'SU-57'", "x": 0.80, "y": 0.20, "w": 0.17, "c": "#8B5CF6"}
]

for b in wire_boxes:
    rect = plt.Rectangle((b["x"], b["y"]), b["w"], 0.26, facecolor=b["c"], alpha=0.35, edgecolor=b["c"], lw=2)
    ax1.add_patch(rect)
    ax1.text(b["x"] + b["w"]/2, b["y"] + 0.13, b["name"], ha='center', va='center', color='#F8FAFC', fontsize=8.5, fontweight='bold')

ax1.text(0.05, 0.92, "Protobuf Tel Akışı (Wire Format Byte Stream):", color='#38BDF8', fontsize=11, fontweight='bold')
ax1.set_title("Protobuf Tel Formatı (Tag + Value / Varint Anatomisi)", fontsize=12, fontweight='bold', color='#F8FAFC', pad=10)

# -----------------------------------------------------------------------------
# PANEL 2: Varint vs ZigZag Kodlama Bayt Boyutu
# -----------------------------------------------------------------------------
ax2 = axs[0, 1]
ax2.set_facecolor('#1E293B')

cases = ['Pozitif Sayı (42)\nVarint', 'Büyük Pozitif (1042)\nVarint', 'Negatif (-350)\nStandart int32 (ŞİŞME!)', 'Negatif (-350)\nsint32 ZigZag (OPTİMUM)']
byte_sizes = [1, 2, 10, 2]
colors_zz = ['#10B981', '#38BDF8', '#EF4444', '#10B981']

bars2 = ax2.bar(cases, byte_sizes, color=colors_zz, width=0.45, edgecolor='#334155', lw=1.5)
for bar, val in zip(bars2, byte_sizes):
    yval = bar.get_height()
    status_txt = f"{val} Bayt"
    if val == 10:
        status_txt += "\n(Two's Complement Tuzağı)"
    elif val == 2 and "ZigZag" in bar.get_x().__str__():
        status_txt += "\n(%80 Kazanç)"
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 0.25, status_txt, ha='center', va='bottom', color='#F8FAFC', fontweight='bold', fontsize=8.5)

ax2.set_ylim(0, 12.5)
ax2.set_ylabel("Teldeki Bayt Sayısı", fontsize=10, color='#94A3B8')
ax2.set_title("Varint ve ZigZag Kodlama Verimliliği (int32 vs sint32)", fontsize=12, fontweight='bold', color='#F8FAFC', pad=10)
ax2.grid(axis='y', linestyle=':', alpha=0.4, color='#64748B')
ax2.tick_params(colors='#94A3B8')

# -----------------------------------------------------------------------------
# PANEL 3: Şema Evrimi ve Bilinmeyen Alan Atlatma
# -----------------------------------------------------------------------------
ax3 = axs[1, 0]
ax3.set_facecolor('#1E293B')

compat_scenarios = ['V1 Alıcı <- V1 Mesaj\n(Temel Paket)', 'V1 Alıcı <- V2 Mesaj\n(Ekstra Tag 9 / Fixed64)', 'Kırıcı Değişiklik\n(Field No Çakışması)']
surv_scores = [100, 100, 0]
colors_compat = ['#10B981', '#38BDF8', '#EF4444']

bars3 = ax3.bar(compat_scenarios, surv_scores, color=colors_compat, width=0.45, edgecolor='#334155', lw=1.5)
for bar, val in zip(bars3, surv_scores):
    yval = bar.get_height()
    lbl = "KUSURSUZ AYRIŞTIRMA" if val == 100 and "V1 Alıcı <- V1" in bar.get_x().__str__() else ("GÜVENLE ATLANDI (UnknownField)" if val == 100 else "BOZULMA / ÇÖKME")
    ax3.text(bar.get_x() + bar.get_width()/2.0, yval + 2.5, f"%{val} Başarı\n{lbl}", ha='center', va='bottom', color='#F8FAFC', fontweight='bold', fontsize=8.5)

ax3.set_ylim(0, 130)
ax3.set_ylabel("Ayrıştırma Başarısı (%)", fontsize=10, color='#94A3B8')
ax3.set_title("Şema Evrimi ve Bilinmeyen Alanları (Unknown Fields) Koruma", fontsize=12, fontweight='bold', color='#F8FAFC', pad=10)
ax3.grid(axis='y', linestyle=':', alpha=0.4, color='#64748B')
ax3.tick_params(colors='#94A3B8')

# -----------------------------------------------------------------------------
# PANEL 4: Tel Boyutu Karşılaştırması
# -----------------------------------------------------------------------------
ax4 = axs[1, 1]
ax4.set_facecolor('#1E293B')

proto_formats = ['JSON (Metin)', 'Raw Struct (C++)', 'Protobuf (Binary)']
proto_sizes = [bench_res['size_json'], bench_res['size_struct'], bench_res['size_proto']]
colors_proto = ['#EF4444', '#F59E0B', '#10B981']

bars4 = ax4.bar(proto_formats, proto_sizes, color=colors_proto, width=0.45, edgecolor='#334155', lw=1.5)
for bar, val in zip(bars4, proto_sizes):
    yval = bar.get_height()
    ratio = val / bench_res['size_proto']
    ax4.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{val} Bayt\n({ratio:.1f}x)", ha='center', va='bottom', color='#F8FAFC', fontweight='bold', fontsize=9.5)

ax4.set_ylim(0, max(proto_sizes) * 1.3)
ax4.set_ylabel("Paket Boyutu (Bayt)", fontsize=10, color='#94A3B8')
ax4.set_title("Tel Boyutu Karşılaştırması (Wire Size)", fontsize=12, fontweight='bold', color='#F8FAFC', pad=10)
ax4.grid(axis='y', linestyle=':', alpha=0.4, color='#64748B')
ax4.tick_params(colors='#94A3B8')

plt.tight_layout()
fig.savefig(PNG_FILE, dpi=300, facecolor='#0F172A', edgecolor='none')
plt.close(fig)

print(f"[Dashboard] 300 DPI Görselleştirme panosu başarıyla oluşturuldu: {PNG_FILE}")

# -----------------------------------------------------------------------------
# KATI DOĞRULAMA TESTLERİ (ASSERTIONS)
# -----------------------------------------------------------------------------
print()
print("[*] GÜN 154 Katı Doğrulama Testleri (Assertions) Başlatılıyor...")

assert os.path.exists(PNG_FILE), "PNG görsel artefaktı bulunamadı!"
png_size = os.path.getsize(PNG_FILE)
print(f"  [+] PNG Varlık ve Boyut Doğrulandı: {png_size:,} bayt.")
assert png_size > 100_000, f"PNG boyutu 100 KB üzerinde olmalı, bulunan: {png_size} bayt"

assert len(raw_int32_varint) == 10, "int32 negatif sayı 10 bayta genişlemelidir!"
assert len(zigzag_varint) == 2, "sint32 negatif sayı 2 bayt tutmalıdır!"
assert 9 in legacy_unknowns, "Bilinmeyen Tag 9 güvenle yakalanmalıdır!"
assert bench_res['size_proto'] < bench_res['size_json'], "Protobuf JSON'dan daha küçük olmalıdır!"

print()
print("[PASS] Gün 154: Tüm Katı Doğrulama Testleri (Assertions) Başarıyla Geçti!")

## 11. Karşılaştırmalı Analiz ve Mimari Karar Matrisi

Serileştirme teknolojilerinin savunma sistemleri açısından karşılaştırma matrisi:

| Kriter | Protocol Buffers (Protobuf) | FlatBuffers (Gün 155) | Raw C-Struct | JSON / XML |
| :--- | :--- | :--- | :--- | :--- |
| **Paket Boyutu** | **En Küçük (Varint + Tag)** | Küçük / Orta (İç dolgu var)| Sabit / Orta | Çok Büyük (%500 Fazla) |
| **Sıfır Kopyalama (Zero-Copy)**| Hayır (Unpack / Parse şart)| **Evet (Doğrudan Bellek Erişimi)**| **Evet (Doğrudan Bellek)** | İmkansız |
| **Diller Arası Şema Desteği** | **Mükemmel (C++, Py, Go, Rust)**| Mükemmel (C++, Py, Rust) | Yalnızca C/C++ | Evrensel Metin |
| **Geriye/İleriye Uyumluluk** | **Mükemmel (Unknown Fields)**| **Mükemmel (Vtable bazlı)** | Çok Zayıf (Ofsetler kilitli)| Dinamik |
| **Tipik Savunma Kullanımı** | Dağıtık C2, gRPC Mikroservis | 1000 Hz Radar, Simülatör | Füze Güdüm, Aviyonik Bus | Web GUI, REST Yapılandırma|

## 12. Donanım ve İşletim Sistemi Seviyesi Etkileşimler

### 1. Protobuf Bit Kaydırma ve CPU Boru Hattı:
Varint çözümü döngüsel olarak `shift` ve `and` işlemleri gerektirir:
```cpp
// Her baytta döngü:
res |= (b & 0x7F) << shift;
if (!(b & 0x80)) break;
```
Modern CPU'lar branch predictor sayesinde tek baytlık küçük tamsayıları çok hızlı çözer. Ancak çok sayıda Varint çözüldüğünde döngü içi dallanmalar (branch misprediction) CPU boru hattında hafif bir gecikme yaratır. Bu nedenle mikrosaniye hassasiyetli uçuş döngülerinde `fixed32` ve `fixed64` gibi sabit boyutlu tipler tercih edilmelidir.

### 2. Little-Endian Donanım Uyumu:
Protobuf standardı `fixed32`, `fixed64`, `float` ve `double` tiplerini her zaman **Little-Endian** olarak kodlar. Bu durum modern x86_64 ve ARM mimarilerinde doğrudan donanım yerleşimi ile uyumludur ve bayt ters çevirme (`bswap`) maliyetini sıfıra indirir.

## 13. Güvenlik, Hata Ayıklama ve Taktik İpuçları

### 1. Wireshark Protobuf Ayrıştırıcı (Dissector) Kullanımı:
Wireshark'ta Protobuf paketlerini ayrıştırmak için:
1. Menüden `Preferences -> Protocols -> ProtoBuf` seçeneğine gidin.
2. `Search paths` kısmına taktik `.proto` dosyalarınızın dizinini ekleyin.
3. İlgili UDP veya TCP portunu (örn: 50051) Protobuf dissector'ına bağlayarak tüm alanları insan gözüyle net şekilde izleyin.

### 2. Güvenlik Sınırları:
Protobuf paketlerinde derin iç içe geçme (recursive embedded messages) veya devasa Varint dizileri ile DoS saldırısı yapılabilir.  
C++'ta `google::protobuf::io::CodedInputStream` üzerinde mutlaka şu sınırlar tanımlanmalıdır:
```cpp
coded_input.SetRecursionLimit(32);
coded_input.SetTotalBytesLimit(10 * 1024 * 1024); // Azami 10 MB
```

## 14. Günün Mini Uygulaması: Taktik Protobuf Dağıtıcısı

Günün mini uygulamasında, bir taktik radar izleme istasyonu ile komuta merkezi aboneleri arasında:
1. `.proto` şemasına sadık kalarak `track_id`, `azimuth`, `radial_vel` ve `classification` alanlarını ikili Varint/Tag formatına dönüştüren,
2. Negatif hızlarda ZigZag kodlamasını kullanarak tel şişmesini önleyen,
3. Gelecek nesil sistemlerden gelen bilinmeyen alanları (Tag 9) çökmeksizin koruyan ve aktaran `TacticalProtobufBroker` mimarisini tasarladık ve doğruladık.

## 15. Sık Yapılan Hatalar ve Anti-Pattern'ler

1. **Alan Numaralarını (Field Number) Değiştirmek:** `.proto` dosyasında alan adını değiştirmek güvenlidir ancak numarasını (`= 1` yerine `= 2`) değiştirmek tüm filoyu felç eder.
2. **Negatif Değerlerde `sint32` Yerine `int32` Kullanmak:** Two's complement genişlemesi yüzünden negatif sayıların 10 bayta şişmesi ve telsiz bandını israf etmesi.
3. **Hard Real-Time Sistemde `SerializeToString` Çağırmak:** Sürekli dinamik heap tahsisi yaparak determinizmi ve gecikmeyi bozmak.
4. **Protobuf'ı Sıfır Kopyalama (Zero-Copy) Sanmak:** Protobuf nesneye ayrıştırma (parse/unpack) gerektirir; sıfır kopyalama gereken radar sinyal işleme döngülerinde FlatBuffers (Gün 155) tercih edilmelidir.

## 16. Mülakat ve Saha Deneyimi Soruları (ASELSAN / HAVELSAN / ROKETSAN / TUSAŞ)

### Soru 1: Protocol Buffers tel formatında bir alanın anahtarı (Tag / Key) nasıl hesaplanır ve telde alan adları neden gönderilmez?
**Cevap:** Protobuf telinde anahtar metinleri (`"azimuth"`) yerine `(field_number << 3) | wire_type` formülüyle üretilen tekil bir Varint tamsayısı (Tag) gönderilir. En alt 3 bit verinin tel tipini (Varint, 64-bit, Length-delimited, 32-bit), üst bitler ise alan numarasını belirtir. Alan adları gönderilmediği için JSON'a kıyasla %70-80 bant genişliği tasarrufu sağlanır ve alıcı taraf string karşılaştırması yapmadan doğrudan alan numarasıyla eşleşir.

### Soru 2: `sint32` tipi ile `int32` arasındaki fark nedir? Negatif hız ve koordinat içeren aviyonik telemetrilerde hangisi neden tercih edilmelidir?
**Cevap:** Standart `int32`, negatif sayıları 64-bit ikiye tümleyen (two's complement) olarak saklar ve Varint kodlamasında işaret biti nedeniyle daima 10 bayt yer kaplar. `sint32` ise ZigZag kodlaması kullanır; negatif ve pozitif sayıları pozitif tamsayılara çapraz eşler ($0 \to 0, -1 \to 1, 1 \to 2, -2 \to 3$). Bu sayede $-350$ gibi küçük negatif sayılar telde 10 bayt yerine yalnızca 2 bayt tutar. Taktik telsiz ve aviyonik hatlarda negatif hız, sapma açısı veya irtifa değişimlerinde mutlaka `sint32`/`sint64` kullanılmalıdır.

### Soru 3: Eski bir C2 yazılımı, yeni bir sensörün gönderdiği ve tanımadığı yeni bir Protobuf alanıyla karşılaştığında ne olur?
**Cevap:** Protobuf ileriye dönük uyumluluk (forward compatibility) ilkesine göre tasarlanmıştır. Eski alıcı tanımadığı alan numarasını gördüğünde, Tag içindeki Wire Type'a bakar: eğer Varint ise bitiş baytına kadar okur; Length-delimited ise uzunluk baytı kadar ileri atlar. Alıcı asla çökmez (`graceful skip`); bilmediği alanı atlar veya `unknown_fields` tamponunda saklayarak mesajı başka düğümlere iletirken bütünlüğünü korur.

## 17. Gün Sonu Özeti ve 3-Cümle Çıkarımı

1. **Protocol Buffers Nedir?:** Dilden ve platformdan bağımsız şema sözleşmesi (`.proto`) üzerinden kod üreten, kompakt ikili tel formatına sahip yüksek performanslı bir serileştirme mekanizmasıdır.
2. **Hangi Problemi Çözer?:** JSON'ın boyut ve CPU parsing şişkinliğini ortadan kaldırırken, C-struct'ların diller arası uyumsuzluk ve ABI hizalama risklerini çözer.
3. **Hangi Trade-off/Hata Sınıfını Getirir?:** Sıfır kopyalama (zero-copy) değildir; ayrıştırma için CPU döngüsü gerektirir ve negatif sayılarda `int32` seçilirse 10 baytlık tel şişmesine yol açar.